<a href="https://colab.research.google.com/github/Stefanpu13/AIClub/blob/master/gpt_demo_tiny.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab: Train a subword tokenizer + train a tiny GPT from scratch (Tiny Shakespeare)
# Fully self-contained: downloads data, trains tokenizer, trains GPT, runs completion.

In [ ]:
!pip -q install tokenizers

import os, math, time, random, urllib.request
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.decoders import ByteLevel as ByteLevelDecoder
from tokenizers.processors import TemplateProcessing

In [ ]:
# -------------------------
# 0) Repro + device
# -------------------------
seed = 42
random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# -------------------------
# 1) Download Tiny Shakespeare
# -------------------------
data_url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
data_path = "tinyshakespeare.txt"
if not os.path.exists(data_path):
    urllib.request.urlretrieve(data_url, data_path)

with open(data_path, "r", encoding="utf-8") as f:
    text = f.read()

print("chars:", len(text))
print("preview:\n", text[:300])

#2) Train a subword tokenizer from scratch (byte-level BPE)


In [ ]:
# -------------------------
# 2) Train a subword tokenizer from scratch (byte-level BPE)
#    - byte-level is robust and GPT-2-like behavior
# -------------------------
special_tokens = ["<pad>", "<bos>", "<eos>", "<unk>"]
vocab_size = <>  # for tinyshakespeare, 2k-8k is fine

tok = Tokenizer(BPE(unk_token="<unk>"))
tok.pre_tokenizer = ByteLevel(add_prefix_space=True)
tok.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=vocab_size,
    min_frequency=2,
    special_tokens=special_tokens,
)

tok.train([data_path], trainer)
# Add consistent BOS/EOS handling
tok.post_processor = TemplateProcessing(
    single="<bos> $A <eos>",
    pair="<bos> $A <eos> $B:1 <eos>:1",
    special_tokens=[
        ("<bos>", tok.token_to_id("<bos>")),
        ("<eos>", tok.token_to_id("<eos>")),
    ],
)

print("trained tokenizer vocab:", tok.get_vocab_size())
print("tokenizer test tokens:", tok.encode("ROMEO:\nWherefore art thou Romeo?").tokens[:30])

#3) Encode dataset into token IDs (one long stream)

In [ ]:
ids = tok.encode(text).ids
ids = torch.tensor(ids, dtype=torch.long)
print("num tokens:", ids.numel())

# Train/val split on token stream
n = ids.numel()
split = int(0.9 * n)
train_ids = ids[:split]
val_ids   = ids[split:]
print("train tokens:", train_ids.numel(), "| val tokens:", val_ids.numel())

In [ ]:
# -------------------------
# 4) Batch sampler (random windows)
# Questions: What does this code block do?
# What is the size of x and y here?
# Why is there a +1 for y?
# -------------------------

def get_batch(data_ids, block_size, batch_size):
    # sample starting positions
    max_i = data_ids.numel() - (block_size + 1)
    ix = torch.randint(0, max_i, (batch_size,))
    x = torch.stack([data_ids[i:i+block_size] for i in ix])
    y = torch.stack([data_ids[i+1:i+block_size+1] for i in ix])
    return x.to(device), y.to(device)

# -------------------------
# 5) Tiny GPT model (from scratch)
# -------------------------
@dataclass
class GPTConfig:
    vocab_size: int
    block_size: int = 128
    n_layer: int = 4
    n_head: int = 4
    n_embd: int = 256
    dropout: float = 0.1

class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.cfg = cfg
        self.n_head = cfg.n_head
        self.head_dim = cfg.n_embd // cfg.n_head

        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)
        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

        # causal mask (buffer so it moves with device)
        mask = torch.tril(torch.ones(cfg.block_size, cfg.block_size)).view(1, 1, cfg.block_size, cfg.block_size)
        self.register_buffer("mask", mask)

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv(x)  # (B,T,3C)
        q, k, v = qkv.split(C, dim=2)

        # reshape to heads
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)  # (B, nh, T, hd)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)    # (B, nh, T, T)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.dropout(att)

        y = att @ v  # (B, nh, T, hd)
        y = y.transpose(1, 2).contiguous().view(B, T, C)  # (B,T,C)
        y = self.proj(y)
        y = self.dropout(y)
        return y

class MLP(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.fc1 = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.fc2 = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x):
        x = self.fc1(x)
        x = F.gelu(x)          # standard GPT uses GeLU
        x = self.fc2(x)
        x = self.dropout(x)
        return x

class Block(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.wte = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.wpe = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

        # weight tying (common in GPTs)
        self.lm_head.weight = self.wte.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        assert T <= self.cfg.block_size

        pos = torch.arange(0, T, device=idx.device).unsqueeze(0)  # (1,T)
        x = self.wte(idx) + self.wpe(pos)
        x = self.drop(x)
        for blk in self.blocks:
            x = blk(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)  # (B,T,vocab)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

@torch.no_grad()
def estimate_loss(model, block_size, batch_size, eval_iters=50):
    model.eval()
    out = {}
    for split_name, data_ids in [("train", train_ids), ("val", val_ids)]:
        losses = []
        for _ in range(eval_iters):
            xb, yb = get_batch(data_ids, block_size, batch_size)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split_name] = sum(losses) / len(losses)
    model.train()
    return out

@torch.no_grad()
def generate(model, prompt_text, max_new_tokens=200, temperature=1.0, top_k=50):
    model.eval()

    # encode prompt WITHOUT forcing BOS/EOS template; we'll feed as-is
    # so create a "raw" encoding by temporarily removing post_processor usage:
    # easiest: encode and then strip bos/eos if present from template
    enc = tok.encode(prompt_text).ids
    # if template added <bos> and <eos>, try stripping them for prompting
    bos_id = tok.token_to_id("<bos>")
    eos_id = tok.token_to_id("<eos>")
    if len(enc) >= 2 and enc[0] == bos_id and enc[-1] == eos_id:
        enc = enc[1:-1]

    idx = torch.tensor(enc, dtype=torch.long, device=device).unsqueeze(0)  # (1,T)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.cfg.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / max(temperature, 1e-8)

        if top_k is not None:
            v, _ = torch.topk(logits, k=min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float("inf")

        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)  # (1,1)
        idx = torch.cat([idx, next_id], dim=1)

    out_ids = idx[0].tolist()
    return tok.decode(out_ids)